In [ ]:
import os, sys
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"
os.environ["WORKSPACE_CDR"]

In [ ]:
import os, sys

# Make the cohort package importable when the notebook runs from v2/
sys.path.insert(0, os.path.abspath("."))

import pandas as pd
from cohort import CohortConfig, Runner, concepts, queries

runner = Runner()                       # uses WORKSPACE_CDR + ambient creds
cfg = CohortConfig(dataset=runner.dataset)
# cfg.enable_drug_exclusions = True     # optional: add interfering/disrupting-drug exclusions
print("CDR dataset:", cfg.dataset)

In [ ]:
sql = f"""
with tsh_concepts AS (SELECT concept_id FROM UNNEST([3019762, 3019170, 3009201]) AS concept_id),
hypo_diag AS (
    SELECT co.person_id, MIN(co.condition_start_date) AS first_dx_date
    FROM `{cfg.dataset}.condition_occurrence` co
    WHERE
        co.condition_concept_id IN (
            SELECT descendant_concept_id
            FROM `{cfg.dataset}.concept_ancestor`
            WHERE ancestor_concept_id IN (135215, 140673)
        )
        OR co.condition_source_concept_id IN (
            SELECT concept_id FROM `{cfg.dataset}.concept`
            WHERE (vocabulary_id = 'ICD10CM' AND concept_code IN ('E00', 'E00.0', 'E00.1', 'E00.2', 'E00.9', 'E01', 'E01.0', 'E01.1', 'E01.2', 'E01.8', 'E02', 'E03', 'E03.0', 'E03.1', 'E03.2', 'E03.3', 'E03.4', 'E03.8', 'E03.9', 'E06.3'))
               OR (vocabulary_id = 'ICD9CM'  AND concept_code IN ('243', '244', '244.0', '244.1', '244.2', '244.3', '244.8', '244.9'))
        )
    GROUP BY co.person_id
    HAVING COUNT(DISTINCT co.condition_start_date) >= 2
),
t3_first AS (
    SELECT de.person_id, MIN(de.drug_exposure_start_date) AS first_t3_date
    FROM `{cfg.dataset}.drug_exposure` de
    WHERE de.drug_concept_id IN (
        SELECT DISTINCT cd.concept_id
        FROM `{cfg.dataset}.concept_ancestor` ca
        JOIN `{cfg.dataset}.concept` cd ON cd.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = 1505346
          AND cd.domain_id = 'Drug'
          AND cd.standard_concept = 'S'
    )
    GROUP BY de.person_id
),
t4_first AS (
    SELECT de.person_id, MIN(de.drug_exposure_start_date) AS first_t4_date
    FROM `{cfg.dataset}.drug_exposure` de
    WHERE de.drug_concept_id IN (
        SELECT DISTINCT cd.concept_id
        FROM `{cfg.dataset}.concept_ancestor` ca
        JOIN `{cfg.dataset}.concept` cd ON cd.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = 1501700
          AND cd.domain_id = 'Drug'
          AND cd.standard_concept = 'S'
    )
    GROUP BY de.person_id
),
excluded_persons AS (
    -- Structural confounders (celiac/IBD/post-GI-surgery malabsorption,
    -- central hypothyroidism, thyroid cancer): matched by ICD source code, ever.
    SELECT DISTINCT co.person_id
    FROM `{cfg.dataset}.condition_occurrence` co
    WHERE co.condition_source_concept_id IN (
        SELECT concept_id FROM `{cfg.dataset}.concept`
            WHERE (vocabulary_id = 'ICD10CM' AND concept_code IN ('K90.0', 'K90.9', 'K50', 'K50.0', 'K50.1', 'K50.8', 'K50.9', 'K51', 'K51.0', 'K51.2', 'K51.3', 'K51.5', 'K51.8', 'K51.9', 'K91.1', 'K91.2', 'Z90.3', 'Z98.84', 'E23.0', 'E23.1', 'C73'))
               OR (vocabulary_id = 'ICD9CM'  AND concept_code IN ('579.0', '555', '556', '564.2', 'V45.3', '253.2', '253.3', '253.4', '193'))
    )
    UNION DISTINCT
    -- Pregnancy / puerperium within the treatment-response window.
    SELECT DISTINCT co.person_id
    FROM `{cfg.dataset}.condition_occurrence` co
    JOIN t4_first t4 ON co.person_id = t4.person_id
    WHERE co.condition_source_concept_id IN (
        SELECT concept_id FROM `{cfg.dataset}.concept`
            WHERE (vocabulary_id = 'ICD10CM' AND concept_code IN ('Z33.1', 'Z34', 'Z34.0', 'Z34.8', 'Z34.9', 'Z3A', 'O99', 'O99.0', 'O99.1', 'O99.2', 'O99.3'))
               OR (vocabulary_id = 'ICD9CM'  AND concept_code IN ('V22', 'V22.0', 'V22.1', 'V22.2', 'V23', '650'))
    )
      AND co.condition_start_date >= DATE_SUB(t4.first_t4_date, INTERVAL 365 DAY)
      AND co.condition_start_date <= DATE_ADD(t4.first_t4_date, INTERVAL 730 DAY)
),
base_cohort AS (
    SELECT hd.person_id, hd.first_dx_date, t4.first_t4_date, p.birth_datetime
    FROM hypo_diag hd
    JOIN t4_first t4 ON hd.person_id = t4.person_id
    JOIN `{cfg.dataset}.person` p ON p.person_id = hd.person_id
),
bc_t4_start as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from base_cohort
    WHERE
        -- T4 starts within [dx, dx + window], never before diagnosis
        first_t4_date >= first_dx_date
        AND first_t4_date <= DATE_ADD(first_dx_date, INTERVAL 365 DAY)
),
bc_adult as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from bc_t4_start
    WHERE
        -- adult at treatment start
        DATE_DIFF(first_t4_date, DATE(birth_datetime), YEAR) >= 18
),
bc_high_tsh as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from bc_adult bc
    WHERE
        -- elevated pre-treatment TSH confirming genuine hypothyroidism
        EXISTS (
            SELECT 1 FROM `{cfg.dataset}.measurement` m
            JOIN tsh_concepts tc ON m.measurement_concept_id = tc.concept_id
            WHERE m.person_id = bc.person_id
              AND m.value_as_number > 4.2
              AND m.measurement_date >= DATE_SUB(bc.first_dx_date, INTERVAL 365 DAY)
              AND m.measurement_date <= bc.first_dx_date
        )
),
bc_post_tsh as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from bc_high_tsh bc
    WHERE
        -- at least one on-treatment TSH available at steady state (3-6 mo)
        EXISTS (
            SELECT 1 FROM `{cfg.dataset}.measurement` m
            JOIN tsh_concepts tc ON m.measurement_concept_id = tc.concept_id
            WHERE m.person_id = bc.person_id
              AND m.value_as_number IS NOT NULL
              AND m.measurement_date >= DATE_ADD(bc.first_t4_date, INTERVAL 90 DAY)
              AND m.measurement_date <= DATE_ADD(bc.first_t4_date, INTERVAL 180 DAY)
        )
),
bc_no_t3 as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from bc_post_tsh bc
    WHERE
        -- T3 not co-started within the exclusion window of T4
        NOT EXISTS (
            SELECT 1 FROM t3_first t3
            WHERE t3.person_id = bc.person_id
              AND t3.first_t3_date <= DATE_ADD(bc.first_t4_date, INTERVAL 90 DAY)
        )
),
bc_exclude as (
    select person_id, first_dx_date, first_t4_date, birth_datetime
    from bc_no_t3 bc
    WHERE
        -- not confounded by an excluded condition/drug
        bc.person_id NOT IN (SELECT person_id FROM excluded_persons)
)
select
    (select count(*) from base_cohort) n_base,
    (select count(*) from bc_t4_start) n_t4_start,
    (select count(*) from bc_adult) n_adult,
    (select count(*) from bc_high_tsh) n_high_tsh_start,
    (select count(*) from bc_post_tsh) n_post_tsh,
    (select count(*) from bc_no_t3) n_no_t3,
    (select count(*) from bc_exclude) n_after_excluded
"""

df = runner.df(sql)

df
